<a href="https://colab.research.google.com/github/jahnavichekka/Robot-Arm-Position-Calculator-/blob/main/Robot_Arm_Position_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr


# ============================================================
# ROBOT ARM POSITION CALCULATOR
# Forward Kinematics - 2 Link Robot Arm
# ============================================================

def calculate_robot(arm1, arm2, angle1, angle2):

    # Convert degrees to radians
    theta1 = np.radians(angle1)
    theta2 = np.radians(angle2)

    # Joint 1 coordinates
    x1 = arm1 * np.cos(theta1)
    y1 = arm1 * np.sin(theta1)

    # Joint 2 / End-effector coordinates
    total_angle = theta1 + theta2

    x2 = x1 + arm2 * np.cos(total_angle)
    y2 = y1 + arm2 * np.sin(total_angle)

    # Distance of end effector from base
    distance = np.sqrt(x2**2 + y2**2)

    # Maximum and minimum possible reach
    max_reach = arm1 + arm2
    min_reach = abs(arm1 - arm2)

    # Check whether calculated position is physically reachable
    reachable = min_reach <= distance <= max_reach

    if reachable:
        status = "✅ Position is reachable"
    else:
        status = "❌ Position is outside the reachable workspace"

    # --------------------------------------------------------
    # Create Robot Arm Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(7, 7))

    # Robot arm
    ax.plot(
        [0, x1, x2],
        [0, y1, y2],
        marker='o',
        markersize=12,
        linewidth=5
    )

    # Base
    ax.scatter(
        0, 0,
        s=180,
        marker='s',
        label="Base"
    )

    # End effector
    ax.scatter(
        x2, y2,
        s=180,
        marker='*',
        label="End Effector"
    )

    # Coordinate labels
    ax.annotate(
        f"Joint 1\n({x1:.2f}, {y1:.2f})",
        (x1, y1),
        xytext=(10, 10),
        textcoords="offset points"
    )

    ax.annotate(
        f"End Effector\n({x2:.2f}, {y2:.2f})",
        (x2, y2),
        xytext=(10, 10),
        textcoords="offset points"
    )

    # Calculate graph range
    graph_limit = max(
        arm1 + arm2 + 5,
        10
    )

    ax.set_xlim(-graph_limit, graph_limit)
    ax.set_ylim(-graph_limit, graph_limit)

    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")

    ax.set_title(
        "Robot Arm Position Calculator\n"
        "Forward Kinematics"
    )

    ax.grid(True, alpha=0.3)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_aspect("equal", adjustable="box")

    ax.legend()

    # --------------------------------------------------------
    # Result Information
    # --------------------------------------------------------

    result = f"""
## 🤖 Robot Arm Calculation Result

### 📍 End-Effector Position

**X = {x2:.2f} units**

**Y = {y2:.2f} units**

---

### 🔗 Joint 1 Position

**X = {x1:.2f}**

**Y = {y1:.2f}**

---

### 📐 Robot Information

**Arm 1 Length:** {arm1:.2f}

**Arm 2 Length:** {arm2:.2f}

**Joint 1 Angle:** {angle1:.2f}°

**Joint 2 Relative Angle:** {angle2:.2f}°

**Total End-Effector Angle:** {(angle1 + angle2):.2f}°

---

### 📏 Reachability

**Current distance from base:** {distance:.2f}

**Maximum reach:** {max_reach:.2f}

**Minimum reach:** {min_reach:.2f}

### Status

{status}
"""

    return fig, result


# ============================================================
# RESET FUNCTION
# ============================================================

def reset_values():
    return 10, 8, 30, 45


# ============================================================
# GRADIO USER INTERFACE
# ============================================================

with gr.Blocks(title="Robot Arm Position Calculator") as app:

    gr.Markdown(
        """
        # 🤖 Robot Arm Position Calculator

        ### Forward Kinematics Simulation

        Enter the robot arm lengths and joint angles to calculate
        the exact position of the robot's end-effector.
        """
    )

    with gr.Row():

        # ----------------------------------------------------
        # INPUT SECTION
        # ----------------------------------------------------

        with gr.Column():

            gr.Markdown("## ⚙️ Robot Parameters")

            arm1_input = gr.Number(
                value=10,
                label="Arm 1 Length",
                minimum=0.1
            )

            arm2_input = gr.Number(
                value=8,
                label="Arm 2 Length",
                minimum=0.1
            )

            angle1_input = gr.Slider(
                minimum=-180,
                maximum=180,
                value=30,
                step=1,
                label="Joint 1 Angle (°)"
            )

            angle2_input = gr.Slider(
                minimum=-180,
                maximum=180,
                value=45,
                step=1,
                label="Joint 2 Relative Angle (°)"
            )

            with gr.Row():

                calculate_button = gr.Button(
                    "🚀 Calculate Position",
                    variant="primary"
                )

                reset_button = gr.Button(
                    "🔄 Reset"
                )

        # ----------------------------------------------------
        # OUTPUT SECTION
        # ----------------------------------------------------

        with gr.Column():

            output_plot = gr.Plot(
                label="Robot Arm Visualization"
            )

            output_result = gr.Markdown()


    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    calculate_button.click(
        fn=calculate_robot,
        inputs=[
            arm1_input,
            arm2_input,
            angle1_input,
            angle2_input
        ],
        outputs=[
            output_plot,
            output_result
        ]
    )


    # --------------------------------------------------------
    # RESET ACTION
    # --------------------------------------------------------

    reset_button.click(
        fn=reset_values,
        inputs=[],
        outputs=[
            arm1_input,
            arm2_input,
            angle1_input,
            angle2_input
        ]
    )


    # --------------------------------------------------------
    # INFORMATION SECTION
    # --------------------------------------------------------

    gr.Markdown(
        """
        ---

        ## 📚 How does it work?

        This application uses **Forward Kinematics**.

        The position of the end-effector is calculated using:

        **X = L₁ cos(θ₁) + L₂ cos(θ₁ + θ₂)**

        **Y = L₁ sin(θ₁) + L₂ sin(θ₁ + θ₂)**

        Where:

        - **L₁** = Length of first arm
        - **L₂** = Length of second arm
        - **θ₁** = First joint angle
        - **θ₂** = Second joint angle
        - **X, Y** = End-effector position

        ### 🌎 Real-world applications

        - 🏭 Industrial robotic arms
        - 📦 Warehouse automation
        - 🚗 Automobile manufacturing
        - 🏥 Medical robotics
        - 🔧 Assembly systems
        - 🚀 Space robotics
        """
    )


# ============================================================
# LAUNCH APPLICATION
# ============================================================

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b2811fd1f5ffd2333.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
